# 02 — MLP (capas densas)
Perceptrón multicapa sobre X aplanado `(N, V_in × 23)`.
Catálogo activo: **mlp_s** (64 neuronas). Descomentar `[EXTENDER]` para añadir más variantes.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, yfinance as yf
warnings.simplefilter('ignore')

from keras import Sequential, Input
from keras.layers import Dense

from utils import (TICKERS, INPUT_WINDOWS, OUTPUT_WINDOWS,
                   create_time_series_data, make_splits, eval_mae,
                   get_callbacks, compile_model,
                   plot_history, plot_mae_matrix, build_results_df)

In [ ]:
# ── HIPERPARÁMETROS ───────────────────────────────────────────
EPOCHS     = 100
BATCH_SIZE = 64
QUICK_MODE = False   # True → EPOCHS=20 para pruebas rápidas (~10 min)
if QUICK_MODE: EPOCHS = 20

# Carga de datos
precios = yf.download(TICKERS, start='1945-01-01', auto_adjust=True, progress=False)['Close']
precios.dropna(axis=1, inplace=True)
returns = np.log(precios).diff().dropna()
print(f'Retornos: {returns.shape}  |  EPOCHS={EPOCHS}  BATCH={BATCH_SIZE}')

## Catálogo de arquitecturas MLP
Añadir más modelos descomentando las líneas `[EXTENDER]`.

In [ ]:
# Input shape para MLP: (V_in * 23,) — X debe aplanarse antes de entrenar
def build_mlp(V_in, units, hidden=1):
    """Construye un MLP con `hidden` capas ocultas de `units` neuronas."""
    layers = [Input((V_in * 23,))]
    for _ in range(hidden):
        layers.append(Dense(units, activation='relu'))
    layers.append(Dense(23))   # 23 salidas = 23 activos
    return compile_model(Sequential(layers))

MODELOS = {
    'mlp_s': lambda V: build_mlp(V, units=64,  hidden=1),   # 1 capa × 64 neuronas
    # 'mlp_m': lambda V: build_mlp(V, units=128, hidden=2),  # [EXTENDER] 2 × 128
    # 'mlp_l': lambda V: build_mlp(V, units=256, hidden=2),  # [EXTENDER] 2 × 256
}

## Entrenamiento — bucle sobre todas las ventanas

In [ ]:
results = {}   # { (nombre, V_in, V_out): {'train','val','test','params'} }
historiales = {}

for nombre, build_fn in MODELOS.items():
    for V_in in INPUT_WINDOWS:
        for V_out in OUTPUT_WINDOWS:
            X, y = create_time_series_data(returns, V_in, V_out)
            X_tr, X_v, X_ts, y_tr, y_v, y_ts = make_splits(X, y)

            # MLP necesita X aplanado: (N, V_in*23)
            X_tr_f = X_tr.reshape(len(X_tr), -1)
            X_v_f  = X_v.reshape(len(X_v),  -1)
            X_ts_f = X_ts.reshape(len(X_ts), -1)

            model = build_fn(V_in)
            hist  = model.fit(X_tr_f, y_tr,
                              validation_data=(X_v_f, y_v),
                              epochs=EPOCHS, batch_size=BATCH_SIZE,
                              callbacks=get_callbacks(), verbose=0)

            key = (nombre, V_in, V_out)
            results[key]    = {'train':  eval_mae(model, X_tr_f, y_tr),
                               'val':    eval_mae(model, X_v_f,  y_v),
                               'test':   eval_mae(model, X_ts_f, y_ts),
                               'params': model.count_params()}
            historiales[key] = hist
            print(f'{nombre}  in={V_in:2d}  out={V_out:2d}  '
                  f'train={results[key]["train"]:.4f}  '
                  f'val={results[key]["val"]:.4f}  '
                  f'test={results[key]["test"]:.4f}  '
                  f'epochs={len(hist.history["loss"])}')

## Curvas de entrenamiento

In [ ]:
# Mostrar curvas para el primer modelo activo
nombre = list(MODELOS.keys())[0]
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for i, V_in in enumerate(INPUT_WINDOWS):
    for j, V_out in enumerate(OUTPUT_WINDOWS):
        hist = historiales[(nombre, V_in, V_out)]
        ax = axes[i][j]
        ax.plot(hist.history['loss'],     label='train')
        ax.plot(hist.history['val_loss'], label='val')
        ax.set_title(f'in={V_in} out={V_out}', fontsize=8)
        ax.legend(fontsize=6); ax.tick_params(labelsize=6)
plt.suptitle(f'Curvas de convergencia — {nombre}', fontsize=12)
plt.tight_layout(); plt.show()

## Resultados — MAE en test

In [ ]:
df_res = build_results_df(results)
print(df_res.to_string())

In [ ]:
# Heatmap 4×4 del mejor MAE en test por combinación
from utils import best_per_window
mat = best_per_window(df_res, metric='test')
plot_mae_matrix(mat, title='MLP — mejor MAE test por combinación')